In [ ]:
# #final  task4

# """
# eval_pf_willow.py - Enhanced with detailed metrics
# Evaluation script for PF-Willow dataset using fine-tuned models.
# Uses EXACT same methods as the original SPair-71k evaluation.
# """

# import torch
# import os
# import sys
# import pandas as pd
# import numpy as np
# from PIL import Image
# from pathlib import Path
# from torchvision import transforms
# from torch.utils.data import Dataset
# import torch.nn.functional as F
# from tqdm import tqdm
# import matplotlib.pyplot as plt
# import itertools
# import zipfile
# import cv2

# # Add necessary paths
# sys.path.append(os.path.abspath('dinov2'))
# sys.path.append(os.path.abspath('dinov3'))

# from segment_anything import SamPredictor, sam_model_registry


# def extract_pf_willow(zip_path, extract_to='/content/pf-willow'):
#     """Extract PF-Willow dataset from zip file."""
#     if os.path.exists(extract_to) and os.path.isdir(extract_to):
#         print(f"✓ PF-Willow already extracted at: {extract_to}")
#         return extract_to

#     print(f"Extracting PF-Willow dataset...")
#     print(f"  From: {zip_path}")
#     print(f"  To: {extract_to}")

#     os.makedirs(extract_to, exist_ok=True)

#     with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#         zip_ref.extractall(extract_to)

#     print(f"✓ Extraction complete!\n")
#     return extract_to


# IMAGE_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"]
# ANNO_EXTS  = [".mat", ".txt", ".pts"]

# def _first_existing(base: Path, stem_or_rel: str, exts):
#     """Find first existing file with given stem and extensions."""
#     p = base / stem_or_rel
#     if p.exists() and p.is_file():
#         return p

#     if p.suffix:
#         if p.exists():
#             return p
#         p_nosuffix = p.with_suffix('')
#         for ext in exts:
#             cand = base / f"{p_nosuffix.name}{ext}"
#             if cand.exists():
#                 return cand
#         raise FileNotFoundError(f"Missing file: {p}")

#     for ext in exts:
#         cand = base / f"{stem_or_rel}{ext}"
#         if cand.exists():
#             return cand
#     raise FileNotFoundError(f"Missing file for '{stem_or_rel}' under {base} with exts {exts}")

# def _load_keypoints_any(anno_path: Path):
#     """Load keypoints from .mat, .txt, or .pts files."""
#     suffix = anno_path.suffix.lower()

#     if suffix == ".mat":
#         import scipy.io as sio
#         mat = sio.loadmat(str(anno_path))

#         for key in ["pts_coord", "pts", "kps", "keypoints"]:
#             if key in mat:
#                 arr = mat[key]
#                 arr = np.squeeze(arr)

#                 if arr.ndim == 2 and arr.shape[0] == 2:
#                     xy = arr.T
#                 elif arr.ndim == 2 and arr.shape[1] == 2:
#                     xy = arr
#                 else:
#                     continue

#                 vis = np.ones((xy.shape[0], 1), dtype=np.float32)
#                 return np.concatenate([xy.astype(np.float32), vis], axis=1)

#         raise KeyError(f"No supported keypoints array found in {anno_path}. Keys: {list(mat.keys())}")

#     lines = anno_path.read_text().strip().splitlines()
#     kps = []
#     for ln in lines:
#         parts = ln.strip().split()
#         if len(parts) >= 2:
#             x = float(parts[0]); y = float(parts[1])
#             v = int(parts[2]) if len(parts) > 2 else 1
#             kps.append([x, y, v])
#     return np.array(kps, dtype=np.float32)


# class PFWillowDataset(Dataset):
#     def __init__(self, root_dir):
#         self.root_dir = Path(root_dir)

#         # Locate PF-dataset folder
#         if (self.root_dir / "PF-dataset").exists():
#             self.dataset_dir = self.root_dir / "PF-dataset"
#         elif self.root_dir.name == "PF-dataset":
#             self.dataset_dir = self.root_dir
#         else:
#             found = list(self.root_dir.rglob("PF-dataset"))
#             if not found:
#                 raise FileNotFoundError(f"PF-dataset not found under: {self.root_dir}")
#             self.dataset_dir = found[0]

#         csv_candidates = []
#         csv_candidates += list(self.dataset_dir.parent.glob("test_pairs.csv"))
#         csv_candidates += list(self.dataset_dir.glob("test_pairs.csv"))
#         csv_candidates += list(self.root_dir.glob("test_pairs.csv"))

#         self.pairs = []
#         self.mode = None

#         if csv_candidates:
#             try:
#                 self.mode = "csv"
#                 self._load_pairs_from_csv(csv_candidates[0])
#             except Exception as e:
#                 print(f"Warning: CSV loading failed ({e}), falling back to folder enumeration")
#                 self.mode = "folder"
#                 self.pairs = []
#                 self._load_pairs_from_folders()
#         else:
#             self.mode = "folder"
#             self._load_pairs_from_folders()

#         print(f"Using PF-dataset at: {self.dataset_dir}")
#         print(f"Pairs loaded: {len(self.pairs)} (mode={self.mode})")

#     def _load_pairs_from_csv(self, csv_path: Path):
#         """Load image pairs from CSV file."""
#         df = pd.read_csv(csv_path, header=None)
#         df = df.dropna(axis=1, how="all")

#         start_idx = 0
#         for i in range(min(3, len(df))):
#             first_val = str(df.iloc[i, 0]).lower()
#             if any(word in first_val for word in ['source', 'src', 'image', 'pair']):
#                 start_idx = i + 1

#         if start_idx > 0:
#             df = df.iloc[start_idx:].reset_index(drop=True)

#         ncols = df.shape[1]
#         if ncols < 2:
#             raise ValueError(f"CSV has <2 columns: {csv_path}")

#         numeric = None
#         if ncols > 2:
#             numeric = df.iloc[:, 2:].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)

#         for i in range(len(df)):
#             try:
#                 src_rel = str(df.iloc[i, 0]).strip().lstrip("./")
#                 trg_rel = str(df.iloc[i, 1]).strip().lstrip("./")

#                 if not src_rel or not trg_rel or src_rel == 'nan' or trg_rel == 'nan':
#                     continue

#                 if src_rel.startswith("PF-dataset/"):
#                     src_rel = src_rel[len("PF-dataset/"):]
#                 if trg_rel.startswith("PF-dataset/"):
#                     trg_rel = trg_rel[len("PF-dataset/"):]

#                 src_img_path = _first_existing(self.dataset_dir, src_rel, IMAGE_EXTS)
#                 trg_img_path = _first_existing(self.dataset_dir, trg_rel, IMAGE_EXTS)

#                 src_kps = trg_kps = None
#                 if numeric is not None:
#                     row = numeric[i]
#                     if not np.all(np.isnan(row)) and row.size % 4 == 0 and row.size >= 8:
#                         K = row.size // 4
#                         src_xy = row[:2*K].reshape(K, 2)
#                         trg_xy = row[2*K:4*K].reshape(K, 2)
#                         vis = np.ones((K, 1), dtype=np.float32)
#                         src_kps = np.concatenate([src_xy, vis], axis=1)
#                         trg_kps = np.concatenate([trg_xy, vis], axis=1)

#                 self.pairs.append({
#                     "src_img_path": src_img_path,
#                     "trg_img_path": trg_img_path,
#                     "src_kps": src_kps,
#                     "trg_kps": trg_kps,
#                     "category": src_img_path.parent.name,
#                 })
#             except (FileNotFoundError, Exception):
#                 continue

#     def _load_pairs_from_folders(self):
#         """Generate all pairs from category folders."""
#         categories = sorted([d for d in self.dataset_dir.iterdir()
#                            if d.is_dir() and not d.name.startswith(".")])

#         for cat_dir in categories:
#             items = []
#             for img_ext in IMAGE_EXTS:
#                 for img_path in cat_dir.glob(f"*{img_ext}"):
#                     stem = img_path.stem
#                     try:
#                         anno_path = _first_existing(cat_dir, stem, ANNO_EXTS)
#                         items.append((img_path, anno_path))
#                     except FileNotFoundError:
#                         pass

#             if len(items) < 2:
#                 continue

#             for a in range(len(items)):
#                 for b in range(a + 1, len(items)):
#                     self.pairs.append({
#                         "src_img_path": items[a][0],
#                         "trg_img_path": items[b][0],
#                         "src_anno_path": items[a][1],
#                         "trg_anno_path": items[b][1],
#                         "src_kps": None,
#                         "trg_kps": None,
#                         "category": cat_dir.name,
#                     })

#     def __len__(self):
#         return len(self.pairs)

#     def __getitem__(self, idx):
#         p = self.pairs[idx]

#         src_img = Image.open(p["src_img_path"]).convert("RGB")
#         trg_img = Image.open(p["trg_img_path"]).convert("RGB")

#         src_size = (src_img.height, src_img.width)
#         trg_size = (trg_img.height, trg_img.width)

#         if p.get("src_kps") is not None and p.get("trg_kps") is not None:
#             src_kps = p["src_kps"]
#             trg_kps = p["trg_kps"]
#         else:
#             src_kps = _load_keypoints_any(p["src_anno_path"])
#             trg_kps = _load_keypoints_any(p["trg_anno_path"])

#         return {
#             "src_img": src_img,
#             "trg_img": trg_img,
#             "src_kps": src_kps,
#             "trg_kps": trg_kps,
#             "src_size": src_size,
#             "trg_size": trg_size,
#             "category": p.get("category", "unknown"),
#             "pair_idx": idx,
#             "src_name": p["src_img_path"].name,
#             "trg_name": p["trg_img_path"].name,
#         }

# def softargmax_2d(similarity_map, temperature=0.01):
#     """Apply 2D soft-argmax for sub-pixel prediction."""
#     H, W = similarity_map.shape
#     sim_flat = similarity_map.reshape(-1)
#     probs = F.softmax(sim_flat / temperature, dim=0)

#     x_coords = torch.arange(W, device=similarity_map.device).float()
#     y_coords = torch.arange(H, device=similarity_map.device).float()
#     grid_x, grid_y = torch.meshgrid(x_coords, y_coords, indexing='xy')

#     grid_x_flat = grid_x.reshape(-1)
#     grid_y_flat = grid_y.reshape(-1)

#     pred_x = (probs * grid_x_flat).sum()
#     pred_y = (probs * grid_y_flat).sum()

#     return pred_x, pred_y


# def evaluate_pf_willow(model, dataset, model_type='dino',
#                        thresholds=[0.05, 0.1, 0.15, 0.2],
#                        img_size=(512, 512),
#                        use_softargmax=False):
#     """
#     Evaluate model on PF-Willow dataset.
#     Returns per-keypoint, per-image, and per-category metrics.
#     """
#     device = 'cuda' if torch.cuda.is_available() else 'cpu'
#     is_sam = (model_type == 'sam')

#     transform = transforms.Compose([
#         transforms.Resize(img_size),
#         transforms.ToTensor(),
#         transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
#     ])

#     # Per-keypoint tracking
#     correct_kps = {t: 0 for t in thresholds}
#     total_kps = 0
#     category_correct = {t: {} for t in thresholds}
#     category_total = {}

#     # Per-image tracking
#     per_image_results = []

#     for idx in tqdm(range(len(dataset)), desc="Evaluating"):
#         try:
#             sample = dataset[idx]
#         except Exception as e:
#             continue

#         src_pil = sample['src_img']
#         trg_pil = sample['trg_img']
#         src_kps = sample['src_kps']
#         trg_kps = sample['trg_kps']
#         src_h, src_w = sample['src_size']
#         trg_h, trg_w = sample['trg_size']
#         category = sample['category']

#         if category not in category_total:
#             category_total[category] = 0
#             for t in thresholds:
#                 category_correct[t][category] = 0

#         # Extract Features
#         if is_sam:
#             layer_idx = getattr(model, 'sam_layer_idx', None)
#             f_src = extract_sam_features(model, np.array(src_pil), layer_idx=layer_idx)
#             f_trg = extract_sam_features(model, np.array(trg_pil), layer_idx=layer_idx)
#         else:
#             src_tensor = transform(src_pil).unsqueeze(0).to(device)
#             trg_tensor = transform(trg_pil).unsqueeze(0).to(device)
#             f_src = extract_dino_features(model, src_tensor)
#             f_trg = extract_dino_features(model, trg_tensor)

#         f_src = F.normalize(f_src, dim=1)
#         f_trg = F.normalize(f_trg, dim=1)
#         fh, fw = f_src.shape[2], f_src.shape[3]

#         norm_factor = max(trg_h, trg_w)

#         # Per-image counters
#         image_correct = {t: 0 for t in thresholds}
#         image_total_kps = 0

#         # Evaluate keypoints
#         num_kps = min(len(src_kps), len(trg_kps))
#         for kp_idx in range(num_kps):
#             src_kp = src_kps[kp_idx]
#             trg_kp = trg_kps[kp_idx]

#             if src_kp[2] == 0 or trg_kp[2] == 0:
#                 continue

#             feat_x = int(round(src_kp[0] / src_w * (fw - 1)))
#             feat_y = int(round(src_kp[1] / src_h * (fh - 1)))
#             feat_x = min(max(feat_x, 0), fw - 1)
#             feat_y = min(max(feat_y, 0), fh - 1)

#             target_feat = f_src[:, :, feat_y, feat_x]
#             sim = torch.einsum('nc,nchw->nhw', target_feat, f_trg)[0]

#             if use_softargmax:
#                 sim_up = F.interpolate(
#                     sim.unsqueeze(0).unsqueeze(0),
#                     scale_factor=8,
#                     mode='bicubic',
#                     align_corners=False
#                 ).squeeze()

#                 pred_x_idx, pred_y_idx = softargmax_2d(sim_up, temperature=0.01)
#                 up_h, up_w = sim_up.shape
#                 pred_x = ((pred_x_idx + 0.5) / up_w) * trg_w
#                 pred_y = ((pred_y_idx + 0.5) / up_h) * trg_h
#             else:
#                 flat_idx = sim.argmax().item()
#                 pred_y_idx = flat_idx // fw
#                 pred_x_idx = flat_idx % fw

#                 pred_x = ((pred_x_idx + 0.5) / fw) * trg_w
#                 pred_y = ((pred_y_idx + 0.5) / fh) * trg_h

#             dist = np.sqrt((pred_x - trg_kp[0])**2 + (pred_y - trg_kp[1])**2)

#             total_kps += 1
#             category_total[category] += 1
#             image_total_kps += 1

#             for t in thresholds:
#                 if dist <= (t * norm_factor):
#                     correct_kps[t] += 1
#                     category_correct[t][category] += 1
#                     image_correct[t] += 1

#         # Store per-image result
#         per_image_results.append({
#             'category': category,
#             'src_image': sample['src_name'],
#             'trg_image': sample['trg_name'],
#             'total_kps': image_total_kps,
#             **{f'PCK@{t}': (image_correct[t] / image_total_kps * 100) if image_total_kps > 0 else 0.0
#                for t in thresholds}
#         })

#     # Compute per-keypoint results
#     per_keypoint_overall = {
#         f"PCK@{t}": (correct_kps[t] / total_kps * 100) if total_kps > 0 else 0.0
#         for t in thresholds
#     }

#     per_keypoint_per_category = {
#         category: {
#             f"PCK@{t}": (category_correct[t][category] / category_total[category] * 100)
#             if category_total[category] > 0 else 0.0
#             for t in thresholds
#         }
#         for category in sorted(category_total.keys())
#     }

#     return {
#         'per_keypoint': {
#             'overall': per_keypoint_overall,
#             'per_category': per_keypoint_per_category,
#         },
#         'per_image': per_image_results,
#         'total_keypoints': total_kps,
#         'total_images': len(per_image_results)
#     }


# def print_results(results_all, thresholds=[0.05, 0.1, 0.15, 0.2]):
#     """Print comprehensive results table."""

#     print("\n" + "="*80)
#     print(" "*25 + "PF-WILLOW EVALUATION RESULTS")
#     print("="*80)

#     # Overall PCK (per-keypoint)
#     print("\n" + "="*80)
#     print("OVERALL PCK - PER-KEYPOINT METRIC")
#     print("="*80)
#     print(f"\n{'Model':<15} " + "".join([f"PCK@{t:<7}" for t in thresholds]))
#     print("-"*80)

#     for model_name, results in results_all.items():
#         pck_values = [results['per_keypoint']['overall'][f'PCK@{t}'] for t in thresholds]
#         print(f"{model_name.upper():<15} " + "".join([f"{pck:>10.2f}%" for pck in pck_values]))

#     # Per-category breakdown
#     print("\n" + "="*80)
#     print("PER-CATEGORY PCK@0.1 COMPARISON")
#     print("="*80)

#     categories = list(list(results_all.values())[0]['per_keypoint']['per_category'].keys())
#     print(f"\n{'Category':<20} " + "".join([f"{m.upper():<15}" for m in results_all.keys()]))
#     print("-"*80)

#     for category in categories:
#         row = f"{category:<20} "
#         for model_name, results in results_all.items():
#             pck = results['per_keypoint']['per_category'][category]['PCK@0.1']
#             row += f"{pck:>10.2f}%    "
#         print(row)

#     # Per-image statistics
#     print("\n" + "="*80)
#     print("PER-IMAGE STATISTICS (PCK@0.1)")
#     print("="*80)

#     for model_name, results in results_all.items():
#         image_pcks = [img['PCK@0.1'] for img in results['per_image']]
#         print(f"\n{model_name.upper()}:")
#         print(f"  Mean:   {np.mean(image_pcks):.2f}%")
#         print(f"  Median: {np.median(image_pcks):.2f}%")
#         print(f"  Std:    {np.std(image_pcks):.2f}%")
#         print(f"  Min:    {np.min(image_pcks):.2f}%")
#         print(f"  Max:    {np.max(image_pcks):.2f}%")

#     print("\n" + "="*80 + "\n")


# def plot_results(results_all, thresholds=[0.05, 0.1, 0.15, 0.2]):
#     """Create enhanced visualization with correct thresholds."""

#     n_models = len(results_all)
#     fig = plt.figure(figsize=(18, 5))

#     # Plot 1: PCK curves (per-keypoint)
#     ax1 = plt.subplot(1, 3, 1)
#     colors = ['#2E86AB', '#A23B72', '#F18F01', '#6A994E']

#     for idx, (model_name, results) in enumerate(results_all.items()):
#         pck_values = [results['per_keypoint']['overall'][f'PCK@{t}'] for t in thresholds]
#         ax1.plot(thresholds, pck_values, marker='o', linewidth=3,
#                 markersize=10, label=model_name.upper(), color=colors[idx % len(colors)])

#     ax1.set_xlabel('Threshold', fontsize=13, fontweight='bold')
#     ax1.set_ylabel('PCK (%)', fontsize=13, fontweight='bold')
#     ax1.set_title('Per-Keypoint PCK Performance', fontsize=14, fontweight='bold', pad=15)
#     ax1.legend(fontsize=11, loc='lower right')
#     ax1.grid(True, alpha=0.3, linestyle='--')
#     ax1.set_ylim([0, 100])
#     ax1.set_xticks(thresholds)
#     ax1.set_xticklabels([f'{t:.2f}' for t in thresholds])

#     # Plot 2: Per-category comparison at PCK@0.1
#     ax2 = plt.subplot(1, 3, 2)

#     categories = list(list(results_all.values())[0]['per_keypoint']['per_category'].keys())
#     x = np.arange(len(categories))
#     width = 0.8 / n_models

#     for idx, (model_name, results) in enumerate(results_all.items()):
#         pck_values = [results['per_keypoint']['per_category'][cat]['PCK@0.1'] for cat in categories]
#         offset = (idx - (n_models-1)/2) * width
#         ax2.bar(x + offset, pck_values, width, label=model_name.upper(),
#                color=colors[idx % len(colors)], alpha=0.8)

#     ax2.set_xlabel('Category', fontsize=13, fontweight='bold')
#     ax2.set_ylabel('PCK@0.1 (%)', fontsize=13, fontweight='bold')
#     ax2.set_title('Per-Category PCK@0.1', fontsize=14, fontweight='bold', pad=15)
#     ax2.set_xticks(x)
#     ax2.set_xticklabels(categories, rotation=30, ha='right', fontsize=9)
#     ax2.legend(fontsize=10, loc='upper right')
#     ax2.grid(True, alpha=0.3, linestyle='--', axis='y')
#     ax2.set_ylim([0, 100])

#     # Plot 3: Per-image distribution (box plot)
#     ax3 = plt.subplot(1, 3, 3)

#     image_data = []
#     labels = []
#     for model_name, results in results_all.items():
#         image_pcks = [img['PCK@0.1'] for img in results['per_image']]
#         image_data.append(image_pcks)
#         labels.append(model_name.upper())

#     bp = ax3.boxplot(image_data, labels=labels, patch_artist=True, showmeans=True)

#     for patch, color in zip(bp['boxes'], colors):
#         patch.set_facecolor(color)
#         patch.set_alpha(0.6)

#     ax3.set_ylabel('PCK@0.1 (%)', fontsize=13, fontweight='bold')
#     ax3.set_title('Per-Image PCK@0.1 Distribution', fontsize=14, fontweight='bold', pad=15)
#     ax3.grid(True, alpha=0.3, linestyle='--', axis='y')
#     ax3.set_ylim([0, 100])

#     plt.tight_layout()
#     plt.show()


# def main(pf_willow_zip_path='/content/drive/MyDrive/AML-PROJECT-DATA/dataset/pf-willow.zip',
#          pretrained_dir='/content/drive/MyDrive/AML-PROJECT-DATA/checkpoints',
#          models=['dinov2', 'dinov3', 'sam'],
#          use_softargmax=False):
#     """Main evaluation function."""

#     device = 'cuda' if torch.cuda.is_available() else 'cpu'
#     print(f"\n{'='*70}")
#     print(f"PF-WILLOW EVALUATION - Device: {device}")
#     print(f"Testing with PRETRAINED models (no fine-tuning)")
#     print(f"{'='*70}\n")

#     extract_dir = extract_pf_willow(pf_willow_zip_path)

#     print(f"Loading dataset...")
#     dataset = PFWillowDataset(root_dir=extract_dir)
#     print(f"✓ Dataset ready\n")

#     thresholds = [0.05, 0.1, 0.15, 0.2]
#     results_all = {}

#     # Evaluate DINOv2
#     if 'dinov2' in models:
#         print("="*70)
#         print("EVALUATING: DINOv2 ViT-B/14 (Pretrained)")
#         print("="*70)

#         dinov2_weights = os.path.join(pretrained_dir, 'dinov2_vitb14_pretrain.pth')
#         print(f"Loading pretrained weights from: {dinov2_weights}\n")
#         dinov2 = torch.hub.load('dinov2', 'dinov2_vitb14', source='local', pretrained=dinov2_weights)
#         dinov2 = dinov2.to(device).eval()

#         results_dinov2 = evaluate_pf_willow(
#             dinov2, dataset, model_type='dino',
#             img_size=(518, 518), thresholds=thresholds,
#             use_softargmax=use_softargmax
#         )

#         results_all['dinov2'] = results_dinov2

#     # Evaluate DINOv3
#     if 'dinov3' in models:
#         print("\n" + "="*70)
#         print("EVALUATING: DINOv3 ViT-B/16 (Pretrained)")
#         print("="*70)

#         dinov3_weights = os.path.join(pretrained_dir, 'dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth')
#         print(f"Loading pretrained weights from: {dinov3_weights}\n")
#         dinov3 = torch.hub.load('dinov3', 'dinov3_vitb16', source='local', pretrained=dinov3_weights)
#         dinov3 = dinov3.to(device).eval()

#         results_dinov3 = evaluate_pf_willow(
#             dinov3, dataset, model_type='dino',
#             img_size=(512, 512), thresholds=thresholds,
#             use_softargmax=use_softargmax
#         )

#         results_all['dinov3'] = results_dinov3

#     # Evaluate SAM
#     if 'sam' in models:
#         print("\n" + "="*70)
#         print("EVALUATING: SAM ViT-B (Pretrained)")
#         print("="*70)

#         sam_weights = os.path.join(pretrained_dir, 'sam_vit_b_01ec64.pth')
#         print(f"Loading pretrained weights from: {sam_weights}\n")
#         sam = sam_model_registry["vit_b"](checkpoint=sam_weights)
#         sampredictor = SamPredictor(sam)
#         sampredictor.model = sampredictor.model.to(device).eval()

#         results_sam = evaluate_pf_willow(
#             sampredictor, dataset, model_type='sam',
#             img_size=(512, 512), thresholds=thresholds,
#             use_softargmax=use_softargmax
#         )

#         results_all['sam'] = results_sam

#     # Display results
#     print_results(results_all, thresholds)
#     plot_results(results_all, thresholds)

#     return results_all


# if __name__ == '__main__':
#     results = main(
#         pf_willow_zip_path='/content/drive/MyDrive/AML-PROJECT-DATA/dataset/pf-willow.zip',
#         pretrained_dir='/content/drive/MyDrive/AML-PROJECT-DATA/checkpoints',
#         models=['dinov2', 'dinov3', 'sam'],
#         use_softargmax=False
#     )

TASK 4 - EVALUATION OF FINETUNED MODELS ON PF-WILLOW

In [1]:
# ============================================================================
# TASK 4: EVALUATE FINETUNED MODELS ON PF-WILLOW DATASET
# ============================================================================

# ====================
# SETUP
# ====================

# 1. Clone repositories
!git clone https://github.com/Luffy65/Semantic-Correspondence.git  # Your repo

# uncomment git checkout if needed
# Switch to dev-task4 branch
# %cd Semantic-Correspondence
# !git checkout dev-task4
# %cd ..

!git clone https://github.com/facebookresearch/dinov2.git           # DINOv2
!git clone https://github.com/facebookresearch/dinov3.git           # DINOv3
!pip install git+https://github.com/facebookresearch/segment-anything.git  # SAM

Cloning into 'Semantic-Correspondence'...
remote: Enumerating objects: 368, done.
remote: Counting objects: 100% (160/160), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 368 (delta 94), reused 108 (delta 43), pack-reused 208 (from 1)
Receiving objects: 100% (368/368), 6.99 MiB | 3.71 MiB/s, done.
Resolving deltas: 100% (198/198), done.
Cloning into 'dinov2'...
remote: Enumerating objects: 708, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 708 (delta 26), reused 12 (delta 12), pack-reused 661 (from 4)
Receiving objects: 100% (708/708), 2.93 MiB | 12.24 MiB/s, done.
Resolving deltas: 100% (343/343), done.
Cloning into 'dinov3'...
remote: Enumerating objects: 538, done.
remote: Counting objects: 100% (363/363), done.
remote: Compressing objects: 100% (264/264), done.
remote: Total 538 (delta 201), reused 99 (delta 99), pack-reused 175 (from 1)
Receiving objects: 100% (538/538), 9.88 MiB | 13.53 M

In [2]:

# 2. Install requirements
!pip install -r Semantic-Correspondence/requirements.txt
!pip install -r dinov2/requirements.txt
!pip install -r dinov3/requirements.txt

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu117, https://pypi.nvidia.com
ERROR: Could not find a version that satisfies the requirement torch==2.0.0 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1)
ERROR: No matching distribution found for torch==2.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 55.4 MB/s eta 0:00:00


In [3]:
# ====================
# LIBRARIES
# ====================

import torch
import os
import sys
import cv2
import numpy as np
import json
from PIL import Image
import torch.nn.functional as F
from torchvision import transforms
from tqdm import tqdm
import pandas as pd
from segment_anything import SamPredictor, sam_model_registry
import shutil

In [4]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ====================
# EXTRACT PF-WILLOW DATASET FROM DRIVE
# ====================

# Paths
DRIVE_ROOT = '/content/drive/MyDrive/AML-PROJECT-DATA'
DATASET_ROOT = os.path.join(DRIVE_ROOT, 'dataset')
DATASET_ARCHIVE = os.path.join(DATASET_ROOT, 'pf-willow.zip')
LOCAL_DATA_DIR = '/content/data'

# Extract dataset to local VM
if not os.path.exists(LOCAL_DATA_DIR):
    print(f"Extracting {DATASET_ARCHIVE} to local VM...")
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    shutil.unpack_archive(DATASET_ARCHIVE, LOCAL_DATA_DIR, format='zip')
    print(f"Done! Data is ready at {LOCAL_DATA_DIR}")
else:
    print("Data already loaded.")

PFWILLOW_ROOT = LOCAL_DATA_DIR

Mounted at /content/drive
Extracting /content/drive/MyDrive/AML-PROJECT-DATA/dataset/pf-willow.zip to local VM...
Done! Data is ready at /content/data


In [5]:
# ====================
# IMPORT PF-WILLOW DATASET CLASS
# ====================

# Add Semantic-Correspondence to path
sys.path.insert(0, '/content/Semantic-Correspondence/datasets')

# Import the PFWillowDataset class from your repo
from Pf_willow import PFWillowDataset

MODEL LOADING AND PATH CONFIG

In [6]:
# ====================
# LOAD MODELS WITH FINETUNED WEIGHTS
# ====================

# Paths to your finetuned checkpoints in Google Drive
CHECKPOINT_ROOT = '/content/drive/MyDrive/AML-PROJECT-DATA/checkpoints/finetuned/final'

# Repository directories
DINOV2_REPO_DIR = 'dinov2'
DINOV3_REPO_DIR = 'dinov3'

CHOOSE ONE OF THE CELL IN THIS SECTION TO SELECT A MODEL TO EVALUATE

DINOV2 - FINETUNED

In [ ]:
DINOV2_FINETUNED_PATH = os.path.join(CHECKPOINT_ROOT, 'dinov2_finetuned.pth')
# Load DINOv2 with finetuned weights
dinov2_vitb14 = torch.hub.load(DINOV2_REPO_DIR, 'dinov2_vitb14', source='local', pretrained=False)
dinov2_checkpoint = torch.load(DINOV2_FINETUNED_PATH, map_location='cuda',  weights_only=False)
dinov2_vitb14.load_state_dict(dinov2_checkpoint['model_state_dict'])
dinov2_vitb14.eval()
dinov2_vitb14.cuda()
print(f"✓ DINOv2 loaded from {DINOV2_FINETUNED_PATH}")

DINO V3- FINETUNED

In [8]:
DINOV3_FINETUNED_PATH = os.path.join(CHECKPOINT_ROOT, 'dinov3_finetuned.pth')
# Load DINOv3 with finetuned weights
dinov3_vitb16 = torch.hub.load(DINOV3_REPO_DIR, 'dinov3_vitb16', source='local', pretrained=False)
dinov3_checkpoint = torch.load(DINOV3_FINETUNED_PATH, map_location='cuda', weights_only=False)
dinov3_vitb16.load_state_dict(dinov3_checkpoint['model_state_dict'])
dinov3_vitb16.eval()
dinov3_vitb16.cuda()
print(f"✓ DINOv3 loaded from {DINOV3_FINETUNED_PATH}")

✓ DINOv3 loaded from /content/drive/MyDrive/AML-PROJECT-DATA/checkpoints/finetuned/final/dinov3_finetuned.pth


SAM - FINETUNED

In [ ]:
SAM_FINETUNED_PATH = os.path.join(CHECKPOINT_ROOT, 'sam_finetuned.pth')
# Load SAM with finetuned weights
sam = sam_model_registry['vit_b']()
sam_predictor = SamPredictor(sam)
sam_checkpoint = torch.load(SAM_FINETUNED_PATH, map_location='cuda', weights_only=False)
sam_predictor.model.image_encoder.load_state_dict(sam_checkpoint['model_state_dict'])
sam_predictor.model.cuda()
sam_predictor.model.eval()
print(f"✓ SAM loaded from {SAM_FINETUNED_PATH}")

FEATURE EXTRACTION

In [9]:
# ====================
# FEATURE EXTRACTION FUNCTIONS
# ====================

def extract_dino_features(model, img_tensor):
    """
    Extracts dense features from DINO-like models (ViT).
    Returns: (1, Feature_Dim, H_grid, W_grid)
    """
    model.eval()
    with torch.no_grad():
        if hasattr(model, 'forward_features'):
            out = model.forward_features(img_tensor)
            # Handle dictionary output (common in DINOv2/v3)
            if isinstance(out, dict):
                patch_tokens = out.get("x_norm_patchtokens", out.get("x_norm_patch_tokens"))
            else:
                patch_tokens = out

            if patch_tokens is None:
                raise ValueError(f"Could not find patch tokens. Keys: {out.keys() if isinstance(out, dict) else 'N/A'}")

            # Reshape: (B, N, D) -> (B, D, H, W)
            B, N, D = patch_tokens.shape
            grid_size = int(np.sqrt(N))
            feature_map = patch_tokens.permute(0, 2, 1).reshape(B, D, grid_size, grid_size)
            return feature_map
    return None

def extract_dino_layers(model, img_tensor, layer_ids):
    """
    Extracts features from intermediate DINO layers.
    layer_ids: list of transformer block indices you want (0-based).
    returns dict: {layer_id: feature_map}
    """
    model.eval()
    with torch.no_grad():

        all_layers_outputs = model.get_intermediate_layers(img_tensor, n=len(model.blocks), norm=True)
        out = {}
        for lid in layer_ids:
            patch_tokens = all_layers_outputs[lid]
            B, N_patches, D = patch_tokens.shape

            # Dynamically calculate grid dimensions based on the number of patch tokens
            grid_size = int(np.sqrt(N_patches))
            H_grid = grid_size
            W_grid = grid_size

            if N_patches != H_grid * W_grid:
                raise ValueError(
                    f"Unexpected number of patch tokens for layer {lid}: {N_patches}. "
                    f"N_patches ({N_patches}) is not a perfect square for a grid. "
                    f"Calculated grid size: {H_grid}x{W_grid} = {H_grid*W_grid}."
                )

            # Reshape: (B, N_patches, D) -> (B, D, H_grid, W_grid)
            feature_map = patch_tokens.permute(0, 2, 1).reshape(B, D, H_grid, W_grid)
            out[lid] = feature_map
        return out

def extract_sam_features(predictor, image_np, res=512, layer_idx=None):
    """
    Extract features from SAM's image encoder with support for variable input resolutions.

    SAM was trained with 1024x1024 images (64x64 patches). For smaller resolutions,
    we temporarily swap in interpolated positional embeddings, then restore them.
    """
    device = next(predictor.model.parameters()).device
    encoder = predictor.model.image_encoder

    # Resize to specified resolution
    image_resized = cv2.resize(image_np, (res, res))

    # Convert to tensor
    img_tensor = torch.from_numpy(image_resized).float().permute(2, 0, 1).unsqueeze(0).to(device)

    # Apply SAM's preprocessing
    pixel_mean = torch.tensor([123.675, 116.28, 103.53]).view(-1, 1, 1).to(device)
    pixel_std = torch.tensor([58.395, 57.12, 57.375]).view(-1, 1, 1).to(device)
    img_tensor = (img_tensor - pixel_mean) / pixel_std

    with torch.no_grad():
        # Check if we need to interpolate positional embeddings
        orig_pos_embed = encoder.pos_embed
        new_size = res // 16  # SAM uses patch size of 16

        if orig_pos_embed is not None and orig_pos_embed.shape[1] != new_size:
            # Interpolate: (1, 64, 64, C) -> (1, C, 64, 64) -> interpolate -> (1, new, new, C)
            pos_embed_interp = F.interpolate(
                orig_pos_embed.data.permute(0, 3, 1, 2),
                size=(new_size, new_size),
                mode='bicubic',
                align_corners=False
            ).permute(0, 2, 3, 1)

            # Temporarily swap positional embeddings (wrap as Parameter)
            encoder.pos_embed = torch.nn.Parameter(pos_embed_interp, requires_grad=False)
            features = encoder(img_tensor)
            encoder.pos_embed = orig_pos_embed  # Restore original
        else:
            features = encoder(img_tensor)

    return features

In [19]:
!pip install scipy

# ====================
# COMPUTE PCK - PF WILLOW
# ====================

def computePCKatT_PFWillow(model, dataset_root, layer_id=None, thresholds=[0.05, 0.1, 0.15, 0.2],
                           img_size=(1024, 1024), use_softargmax=False,
                           softargmax_temp=0.01, softargmax_window=5):
    """
    Compute PCK (Percentage of Correct Keypoints) at various thresholds for PF-Willow.
    Adapted from your exact computePCKatT method.

    Key difference from SPair-71k: Uses image diagonal for normalization instead of bbox max(w,h).

    Returns:
        - per_keypoint: PCK computed as (total correct keypoints) / (total keypoints)
        - per_image: PCK computed for each image pair
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"

    is_sam = False
    if "SamPredictor" in str(type(model)):
        is_sam = True
    else:
        model = model.to(device)

    # PF-Willow dataset structure - AUTO-DISCOVER CATEGORIES
    pf_dataset_dir = os.path.join(dataset_root, 'PF-dataset')

    if not os.path.exists(pf_dataset_dir):
        print(f"ERROR: {pf_dataset_dir} does not exist!")
        return {'per_keypoint': {'overall': {f'PCK@{t}': 0.0 for t in thresholds}, 'per_category': {}},
                'per_image': [], 'total_keypoints': 0, 'total_images': 0}

    # Auto-discover category directories (exclude hidden/system dirs)
    all_items = os.listdir(pf_dataset_dir)
    categories = [d for d in all_items
                  if os.path.isdir(os.path.join(pf_dataset_dir, d))
                  and not d.startswith('.')
                  and not d.startswith('_')
                  and d != '__MACOSX']

    if not categories:
        print(f"ERROR: No category directories found in {pf_dataset_dir}")
        print(f"Items in directory: {all_items}")
        return {'per_keypoint': {'overall': {f'PCK@{t}': 0.0 for t in thresholds}, 'per_category': {}},
                'per_image': [], 'total_keypoints': 0, 'total_images': 0}

    print(f"Auto-discovered {len(categories)} categories: {categories}")

    # Generate image pairs
    pair_list = []
    print("Generating PF-Willow image pairs...")

    for category in categories:
        cat_dir = os.path.join(pf_dataset_dir, category)

        # Find all images with annotations
        import glob
        img_files_jpg = glob.glob(os.path.join(cat_dir, '*.jpg'))
        img_files_png = glob.glob(os.path.join(cat_dir, '*.png'))
        img_files = img_files_jpg + img_files_png

        img_names = []
        for img_path in img_files:
            img_name = os.path.splitext(os.path.basename(img_path))[0]
            # Check for .mat or .txt annotation files
            anno_path_mat = os.path.join(cat_dir, f"{img_name}.mat")
            anno_path_txt = os.path.join(cat_dir, f"{img_name}.txt")
            anno_path_pts = os.path.join(cat_dir, f"{img_name}.pts")

            if os.path.exists(anno_path_mat) or os.path.exists(anno_path_txt) or os.path.exists(anno_path_pts):
                img_names.append(img_name)

        print(f"  Category '{category}': Found {len(img_names)} images with annotations")

        # Generate pairs within category
        import random
        random.seed(42)  # For reproducibility
        num_pairs = min(50, len(img_names) * (len(img_names) - 1) // 2) if len(img_names) >= 2 else 0

        for _ in range(num_pairs):
            if len(img_names) < 2:
                break
            src, trg = random.sample(img_names, 2)
            pair_list.append((category, src, trg))

    print(f"Generated {len(pair_list)} image pairs across {len(categories)} categories")

    if len(pair_list) == 0:
        print(f"\nERROR: No image pairs found!")
        return {
            'per_keypoint': {'overall': {f'PCK@{t}': 0.0 for t in thresholds}, 'per_category': {}},
            'per_image': [],
            'total_keypoints': 0,
            'total_images': 0
        }

    # Transform for DINO
    transform = transforms.Compose([
        transforms.Resize(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Helper function to load keypoints from .mat, .txt, or .pts files
    def load_keypoints(anno_path_base, cat_dir, img_name):
        """Load keypoints from .mat (preferred), .txt, or .pts files."""
        import scipy.io as sio

        # Try .mat first (most common in PF-Willow)
        mat_path = os.path.join(cat_dir, f"{img_name}.mat")
        if os.path.exists(mat_path):
            try:
                mat = sio.loadmat(mat_path)
                # Try different possible keys
                for key in ["pts_coord", "pts", "kps", "keypoints"]:
                    if key in mat:
                        arr = mat[key]
                        arr = np.squeeze(arr)
                        if arr.ndim == 2 and arr.shape[0] == 2:
                            xy = arr.T
                        elif arr.ndim == 2 and arr.shape[1] == 2:
                            xy = arr
                        else:
                            continue
                        vis = np.ones((xy.shape[0], 1), dtype=np.float32)
                        kps_array = np.concatenate([xy.astype(np.float32), vis], axis=1)
                        return kps_array.tolist()
            except Exception as e:
                print(f"Warning: Error loading .mat file {mat_path}: {e}")

        # Fall back to .txt
        txt_path = os.path.join(cat_dir, f"{img_name}.txt")
        if os.path.exists(txt_path):
            with open(txt_path, 'r') as f:
                lines = f.readlines()
            keypoints = []
            for line in lines:
                parts = line.strip().split()
                if len(parts) >= 2:
                    x, y = float(parts[0]), float(parts[1])
                    vis = int(parts[2]) if len(parts) > 2 else 1
                    keypoints.append([x, y, vis])
            return keypoints

        # Fall back to .pts
        pts_path = os.path.join(cat_dir, f"{img_name}.pts")
        if os.path.exists(pts_path):
            with open(pts_path, 'r') as f:
                lines = f.readlines()
            keypoints = []
            for line in lines:
                parts = line.strip().split()
                if len(parts) >= 2:
                    x, y = float(parts[0]), float(parts[1])
                    vis = int(parts[2]) if len(parts) > 2 else 1
                    keypoints.append([x, y, vis])
            return keypoints

        return []

    # Per-keypoint tracking
    correct_kps = {t: 0 for t in thresholds}
    total_kps = 0

    # Per-category tracking
    category_correct = {t: {} for t in thresholds}
    category_total = {}

    # Per-image tracking
    per_image_results = []

    # Progress tracking
    num_pairs_processed = 0

    for category, src_name, trg_name in tqdm(pair_list, desc="Evaluating pairs"):
        if category not in category_total:
            category_total[category] = 0
            for t in thresholds:
                category_correct[t][category] = 0

        cat_dir = os.path.join(pf_dataset_dir, category)

        # Load images
        src_img_path = os.path.join(cat_dir, f"{src_name}.jpg")
        if not os.path.exists(src_img_path):
            src_img_path = os.path.join(cat_dir, f"{src_name}.png")

        trg_img_path = os.path.join(cat_dir, f"{trg_name}.jpg")
        if not os.path.exists(trg_img_path):
            trg_img_path = os.path.join(cat_dir, f"{trg_name}.png")

        if not os.path.exists(src_img_path) or not os.path.exists(trg_img_path):
            continue

        src_pil = Image.open(src_img_path).convert('RGB')
        trg_pil = Image.open(trg_img_path).convert('RGB')
        src_w, src_h = src_pil.size
        trg_w, trg_h = trg_pil.size

        # Load keypoints using helper function
        src_kps = load_keypoints(None, cat_dir, src_name)
        trg_kps = load_keypoints(None, cat_dir, trg_name)

        if len(src_kps) == 0 or len(trg_kps) == 0:
            continue

        # Extract Features (YOUR EXACT LOGIC)
        if is_sam:
            f_src = extract_sam_features(model, np.array(src_pil), res=img_size[0], layer_idx=layer_id)
            f_trg = extract_sam_features(model, np.array(trg_pil), res=img_size[0], layer_idx=layer_id)
        else:
            # DINO logic
            src_tensor = transform(src_pil).unsqueeze(0).to(device)
            trg_tensor = transform(trg_pil).unsqueeze(0).to(device)
            if layer_id is None:
                f_src = extract_dino_features(model, src_tensor)
                f_trg = extract_dino_features(model, trg_tensor)
            else:
                f_src = extract_dino_layers(model, src_tensor, [layer_id])[layer_id]
                f_trg = extract_dino_layers(model, trg_tensor, [layer_id])[layer_id]

        f_src = F.normalize(f_src, dim=1)
        f_trg = F.normalize(f_trg, dim=1)
        fh, fw = f_src.shape[2], f_src.shape[3]

        # PF-Willow uses image diagonal for normalization
        norm_factor = np.sqrt(trg_w**2 + trg_h**2)

        # Per-image counters
        image_correct = {t: 0 for t in thresholds}
        image_total_kps = 0

        # Process each keypoint pair
        for idx in range(min(len(src_kps), len(trg_kps))):
            p_src = src_kps[idx]
            p_trg = trg_kps[idx]

            # Skip invisible keypoints
            if len(p_src) < 3 or len(p_trg) < 3:
                continue
            if p_src[2] == 0 or p_trg[2] == 0:
                continue

            # Map Source Point -> Feature Grid (YOUR EXACT LOGIC)
            feat_x = int(round(p_src[0] / src_w * (fw - 1)))
            feat_y = int(round(p_src[1] / src_h * (fh - 1)))
            feat_x = min(max(feat_x, 0), fw - 1)
            feat_y = min(max(feat_y, 0), fh - 1)

            # Get Source Descriptor
            target_feat = f_src[:, :, feat_y, feat_x]

            # Compute Similarity Map (YOUR EXACT LOGIC)
            sim = torch.einsum('nc,nchw->nhw', target_feat, f_trg)[0]  # (H, W)

            # Get original similarity map dimensions
            h, w = sim.shape[-2], sim.shape[-1]

            if use_softargmax:
                # TODO: Add softargmax_2d function if needed
                # For now, use argmax
                flat_idx = sim.argmax()
                pred_y_idx_coarse = flat_idx // w
                pred_x_idx_coarse = flat_idx % w

                pred_x = (((pred_x_idx_coarse.item() if torch.is_tensor(pred_x_idx_coarse) else pred_x_idx_coarse) + 0.5) / w) * trg_w
                pred_y = (((pred_y_idx_coarse.item() if torch.is_tensor(pred_y_idx_coarse) else pred_y_idx_coarse) + 0.5) / h) * trg_h
            else:
                # Standard Hard Argmax (YOUR EXACT LOGIC)
                flat_idx = sim.argmax()
                pred_y_idx_coarse = flat_idx // w
                pred_x_idx_coarse = flat_idx % w

                pred_x = (((pred_x_idx_coarse.item() if torch.is_tensor(pred_x_idx_coarse) else pred_x_idx_coarse) + 0.5) / w) * trg_w
                pred_y = (((pred_y_idx_coarse.item() if torch.is_tensor(pred_y_idx_coarse) else pred_y_idx_coarse) + 0.5) / h) * trg_h

            # Evaluate distance (YOUR EXACT LOGIC)
            pred_x_val = pred_x.item() if torch.is_tensor(pred_x) else pred_x
            pred_y_val = pred_y.item() if torch.is_tensor(pred_y) else pred_y
            dist = np.sqrt((pred_x_val - p_trg[0])**2 + (pred_y_val - p_trg[1])**2)

            total_kps += 1
            category_total[category] += 1
            image_total_kps += 1

            for t in thresholds:
                if dist <= (t * norm_factor):
                    correct_kps[t] += 1
                    category_correct[t][category] += 1
                    image_correct[t] += 1

        # Store per-image result
        per_image_results.append({
            'category': category,
            'src_image': src_name,
            'trg_image': trg_name,
            'total_kps': image_total_kps,
            **{f'PCK@{t}': (image_correct[t] / image_total_kps * 100) if image_total_kps > 0 else 0.0
               for t in thresholds}
        })

        # Print progress every 100 pairs
        num_pairs_processed += 1
        if num_pairs_processed % 100 == 0:
            current_pck = {t: (correct_kps[t] / total_kps * 100) if total_kps > 0 else 0.0 for t in thresholds}
            print(f"\n  [{num_pairs_processed}/{len(pair_list)}] Current PCK: " +
                  " | ".join([f"@{t}={current_pck[t]:.2f}%" for t in thresholds]))

    # Compute per-keypoint results
    per_keypoint_overall = {
        f"PCK@{t}": (correct_kps[t] / total_kps * 100) if total_kps > 0 else 0.0
        for t in thresholds
    }

    per_keypoint_per_category = {
        category: {
            f"PCK@{t}": (category_correct[t][category] / category_total[category] * 100)
            if category_total[category] > 0 else 0.0
            for t in thresholds
        }
        for category in sorted(category_total.keys())
    }

    return {
        'per_keypoint': {
            'overall': per_keypoint_overall,
            'per_category': per_keypoint_per_category,
        },
        'per_image': per_image_results,
        'total_keypoints': total_kps,
        'total_images': len(per_image_results)
    }


In [11]:
# ====================
# PRINT RESULTS FUNCTION
# ====================

def print_results(results, model_name, thresholds):
    """Print results in the same format as your main notebook."""
    print(f"\n{'='*80}")
    print(f"{model_name} - PF-Willow Evaluation Results")
    print(f"{'='*80}")

    # Overall PCK
    print("\n1. OVERALL PER-KEYPOINT PCK")
    print("-" * 80)
    print(f"{'Threshold':<15}", end='')
    for t in thresholds:
        print(f"PCK@{t:<8}", end='  ')
    print()
    print("-" * 80)
    print(f"{'Overall':<15}", end='')
    for t in thresholds:
        pck = results['per_keypoint']['overall'][f'PCK@{t}']
        print(f"{pck:<8.2f}", end='  ')
    print()

    # Per-category PCK
    print("\n2. PER-CATEGORY PCK")
    print("-" * 80)
    print(f"{'Category':<15}", end='')
    for t in thresholds:
        print(f"PCK@{t:<8}", end='  ')
    print()
    print("-" * 80)

    for category in sorted(results['per_keypoint']['per_category'].keys()):
        print(f"{category:<15}", end='')
        for t in thresholds:
            pck = results['per_keypoint']['per_category'][category][f'PCK@{t}']
            print(f"{pck:<8.2f}", end='  ')
        print()

    print(f"\nTotal keypoints: {results['total_keypoints']}")
    print(f"Total image pairs: {results['total_images']}")

    # Save results to Google Drive
    df = pd.DataFrame(results['per_image'])
    output_dir = '/content/drive/MyDrive/AML-PROJECT-DATA/results_task4'
    output_path = os.path.join(output_dir, f"{model_name.replace(' ', '_')}_pfwillow_results.csv")
    os.makedirs(output_dir, exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"\nSaved per-image results to: {output_path}")


RUN EVALUATION - CHOOSE ONE OF THE FOLLOWING CELLS TO RUN THE EVALUATION ON THE SELECTED MODEL

In [12]:
# ====================
# RUN EVALUATIONS
# ====================
THRESHOLD = [0.05, 0.1, 0.15, 0.2]

In [ ]:
# Evaluate DINOv2
print("\n Evaluating DINOv2 Finetuned ")
results_dinov2 = computePCKatT_PFWillow(
    model=dinov2_vitb14,
    dataset_root=PFWILLOW_ROOT,
    layer_id=None,
    thresholds=THRESHOLD,
    img_size=(518, 518),
    use_softargmax=False
)
print_results(results_dinov2, 'DINOv2_Finetuned', THRESHOLD)

In [20]:
# Evaluate DINOv3
print("\n Evaluating DINOv3 Finetuned ")
results_dinov3 = computePCKatT_PFWillow(
    model=dinov3_vitb16,
    dataset_root=PFWILLOW_ROOT,
    layer_id=None,
    thresholds=THRESHOLD,
    img_size=(512, 512),
    use_softargmax=False
)
print_results(results_dinov3, 'DINOv3_Finetuned', THRESHOLD)


 Evaluating DINOv3 Finetuned 
Auto-discovered 10 categories: ['motorbike(S)', 'winebottle(M)', 'car(S)', 'motorbike(G)', 'car(G)', 'car(M)', 'winebottle(wC)', 'duck(S)', 'motorbike(M)', 'winebottle(woC)']
Generating PF-Willow image pairs...
  Category 'motorbike(S)': Found 10 images with annotations
  Category 'winebottle(M)': Found 10 images with annotations
  Category 'car(S)': Found 10 images with annotations
  Category 'motorbike(G)': Found 10 images with annotations
  Category 'car(G)': Found 10 images with annotations
  Category 'car(M)': Found 10 images with annotations
  Category 'winebottle(wC)': Found 10 images with annotations
  Category 'duck(S)': Found 10 images with annotations
  Category 'motorbike(M)': Found 10 images with annotations
  Category 'winebottle(woC)': Found 10 images with annotations
Generated 450 image pairs across 10 categories


Evaluating pairs:  22%|██▏       | 101/450 [00:19<01:01,  5.64it/s]


  [100/450] Current PCK: @0.05=78.40% | @0.1=93.90% | @0.15=97.10% | @0.2=99.20%


Evaluating pairs:  45%|████▍     | 201/450 [00:37<00:43,  5.75it/s]


  [200/450] Current PCK: @0.05=74.25% | @0.1=93.05% | @0.15=97.15% | @0.2=98.55%


...


  [300/450] Current PCK: @0.05=74.93% | @0.1=92.67% | @0.15=96.33% | @0.2=97.83%


Evaluating pairs:  89%|████████▉ | 401/450 [01:12<00:08,  5.61it/s]


  [400/450] Current PCK: @0.05=74.02% | @0.1=93.20% | @0.15=96.95% | @0.2=98.20%


Evaluating pairs: 100%|██████████| 450/450 [01:21<00:00,  5.54it/s]


DINOv3_Finetuned - PF-Willow Evaluation Results

1. OVERALL PER-KEYPOINT PCK
--------------------------------------------------------------------------------
Threshold      PCK@0.05      PCK@0.1       PCK@0.15      PCK@0.2       
--------------------------------------------------------------------------------
Overall        74.93     93.40     97.02     98.33     

2. PER-CATEGORY PCK
--------------------------------------------------------------------------------
Category       PCK@0.05      PCK@0.1       PCK@0.15      PCK@0.2       
--------------------------------------------------------------------------------
car(G)         74.89     95.11     98.44     99.78     
car(M)         75.78     89.56     92.67     94.44     
car(S)         81.11     94.00     96.00     96.67     
duck(S)        76.89     96.44     100.00    100.00    
motorbike(G)   60.44     90.00     97.78     98.67     
motorbike(M)   61.78     92.44     98.22     98.89     
motorbike(S)   73.56     94.89     98.00 

In [ ]:
# Evaluate SAM
print("\n Evaluating SAM Finetuned ")
results_sam = computePCKatT_PFWillow(
    model=sam_predictor,
    dataset_root=PFWILLOW_ROOT,
    layer_id=None,
    thresholds=THRESHOLD,
    img_size=(512, 512),
    use_softargmax=False
)
print_results(results_sam, 'SAM_Finetuned', THRESHOLD)

print("\n" + "="*80)
print("EVALUATION COMPLETE!")
print("="*80)

In [ ]:
# ====================
# COMPARATIVE SUMMARY
# ====================

print("\n" + "="*80)
print("COMPARATIVE SUMMARY")
print("="*80)

summary_data = []
for model_name, results in [
    ('DINOv2_Finetuned', results_dinov2),
    ('DINOv3_Finetuned', results_dinov3),
    ('SAM_Finetuned', results_sam)
]:
    row = {'Model': model_name}
    for t in THRESHOLD:
        pck = results['per_keypoint']['overall'][f'PCK@{t}']
        row[f'PCK@{t}'] = f"{pck:.2f}"
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
print("\n", summary_df.to_string(index=False))

# Save summary to Google Drive
output_dir = '/content/drive/MyDrive/AML-PROJECT-DATA/results_task4'
summary_path = os.path.join(output_dir, 'pfwillow_summary.csv')
os.makedirs(output_dir, exist_ok=True)
summary_df.to_csv(summary_path, index=False)
print(f"\nSummary saved to: {summary_path}")